# 05. utils.py: позиции, перестановки и NABLA

Этот файл содержит функции, а не новую обучаемую модель. Разберём exist, freeze, get_freqs, local_patching/local_merge, fractal_flatten/unflatten, fast_sta_nabla и nablaT_v2.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/05_utils_and_nabla.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Что здесь имеет веса

exist проверяет None, freeze отключает requires_grad, get_freqs строит фиксированные частоты. Они не имеют optimizer state. NABLA строит mask из текущих Q/K: обучаемые проекции остаются в DiT, алгоритм выбора блоков сам не является отдельной сетью.


In [ ]:
from labkit.native import load_dit_class
load_dit_class(torch.float32)
import sys
u=sys.modules['_k5_lab_float32_128.utils']
print('exist(None):',u.exist(None),'frequencies:',u.get_freqs(6))
layer=u.freeze(torch.nn.Linear(3,4)); print('trainable:',sum(p.numel() for p in layer.parameters() if p.requires_grad))


### 2. local_patching — сложить соседей рядом

Латентная решётка хранит пространственные позиции, а attention работает с последовательностью. Для sparse блоков полезно поставить соседей подряд. Пронумеруем клетки, чтобы видеть permutation без случайных значений.


In [ ]:
x=torch.arange(1*4*4).reshape(1,4,4,1)
patches=u.local_patching(x,(1,4,4),(1,2,2))
restored=u.local_merge(patches,(1,4,4),(1,2,2))
print('original:\n',x[0,:,:,0]); print('groups:\n',patches[:,:,0])
assert torch.equal(x,restored)


### 3. fractal_flatten: dense и block ветви

Название не означает изменение значений. Dense flatten просто склеивает T/H/W. В sparse ветви группы 1×8×8 переставляются вместе с RoPE, чтобы позиции остались соответствующими токенам. Проверяем обратную операцию на 16×16.


In [ ]:
x=torch.arange(1*16*16,dtype=torch.float32).reshape(1,16,16,1)
rope=x.clone()
flat,flat_rope=u.fractal_flatten(x,rope,(1,16,16),block_mask=True)
back=u.fractal_unflatten(flat,(1,16,16),block_mask=True)
assert torch.equal(x,back)
print('first 16 reordered indices:',flat[:16,0])


### 4. Простой block selector

Учебная версия NABLA: сжимаем Query/Key по 4 токена, считаем attention между блоками, сохраняем самые вероятные блоки до накопленной массы P. Пересекающий порог блок включаем. Это иллюстрация отбора, не полная implementation fast_sta_nabla/nablaT_v2.


In [ ]:
q=torch.randn(16,8); k=torch.randn(16,8)
qb=q.reshape(4,4,8).mean(1); kb=k.reshape(4,4,8).mean(1)
prob=(qb@kb.T/np.sqrt(8)).softmax(-1)
values,order=prob.sort(-1,descending=True)
P=0.7
keep_sorted=(values.cumsum(-1)-values)<P
keep=torch.zeros_like(keep_sorted).scatter(-1,order,keep_sorted)
print('kept fraction:',keep.float().mean().item())
fig,axes=plt.subplots(1,2); axes[0].imshow(prob); axes[1].imshow(keep); plt.show()


### 5. STA добавляет локальных соседей

Чисто вероятностный отбор может терять близкие пространственные связи. Локальная маска объединяется с adaptive mask. Упрощённая 1D версия ниже гарантирует диагональ и соседей. Настоящий STA использует временно-пространственную геометрию.


In [ ]:
indices=torch.arange(4)
local=(indices[:,None]-indices[None,:]).abs()<=1
combined=keep|local
print('adaptive:\n',keep.int(),'combined:\n',combined.int())


### 6. Где применяется настоящий NABLA

10s checkpoints используют NABLA; выбранный distill 5s использует dense attention. FlexAttention/NABLA в MPS-порт курса не включены. Не называйте замену sparse на dense эквивалентным восстановлением 10s модели без проверки.


In [ ]:
source_excerpt('kandinsky/models/utils.py','def fast_sta_nabla',28)
source_excerpt('kandinsky/models/utils.py','def nablaT_v2',55)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
